In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
from pathlib import Path

project_root = Path.cwd().parent.parent
raw_file = project_root / "data" / "raw" / "online_retail_II.xlsx"

print(raw_file)
print("File exists:", raw_file.exists())

/Users/nithinsaijatoth/Desktop/UK_Retail_Sales_Customer_Analytics/data/raw/online_retail_II.xlsx
File exists: True


In [3]:
excel_file = pd.ExcelFile(raw_file)

excel_file.sheet_names

['Year 2009-2010', 'Year 2010-2011']

In [4]:
df_2009_2010 = pd.read_excel(
    raw_file,
    sheet_name="Year 2009-2010"
)

df_2010_2011 = pd.read_excel(
    raw_file,
    sheet_name="Year 2010-2011"
)

print("2009-2010 rows:", len(df_2009_2010))
print("2010-2011 rows:", len(df_2010_2011))

2009-2010 rows: 525461
2010-2011 rows: 541910


In [5]:
df_2009_2010["SourceSheet"] = "Year 2009-2010"
df_2010_2011["SourceSheet"] = "Year 2010-2011"

In [6]:
df = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

print("Combined rows:", len(df))
print("Combined columns:", len(df.columns))

Combined rows: 1067371
Combined columns: 9


In [7]:
print(df.columns.tolist())

df.head()

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'SourceSheet']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


In [8]:
missing_values = df.isna().sum().sort_values(ascending=False)

missing_values

Customer ID    243007
Description      4382
Invoice             0
StockCode           0
Quantity            0
InvoiceDate         0
Price               0
Country             0
SourceSheet         0
dtype: int64

In [9]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 12133


In [10]:
cancelled_invoices = df[
    df["Invoice"].astype(str).str.startswith("C", na=False)
]

print("Cancelled invoice rows:", len(cancelled_invoices))

Cancelled invoice rows: 19494


In [11]:
negative_quantity = df[df["Quantity"] < 0]

print("Negative quantity rows:", len(negative_quantity))

Negative quantity rows: 22950


In [12]:
zero_price = df[df["Price"] == 0]
negative_price = df[df["Price"] < 0]

print("Zero price rows:", len(zero_price))
print("Negative price rows:", len(negative_price))

Zero price rows: 6202
Negative price rows: 5


In [13]:
missing_customer_ids = df["Customer ID"].isna().sum()

print("Missing Customer IDs:", missing_customer_ids)

Missing Customer IDs: 243007


In [14]:
missing_descriptions = df["Description"].isna().sum()

print("Missing descriptions:", missing_descriptions)

Missing descriptions: 4382


In [15]:
cancelled_negative = df[
    df["Invoice"].astype(str).str.startswith("C", na=False)
    & (df["Quantity"] < 0)
]

print("Cancelled rows with negative quantity:", len(cancelled_negative))

Cancelled rows with negative quantity: 19493


In [16]:
profile_by_sheet = df.groupby("SourceSheet").agg(
    rows=("Invoice", "size"),
    missing_customer_ids=("Customer ID", lambda x: x.isna().sum()),
    negative_quantity=("Quantity", lambda x: (x < 0).sum()),
    zero_price=("Price", lambda x: (x == 0).sum()),
    negative_price=("Price", lambda x: (x < 0).sum())
)

profile_by_sheet

,rows,missing_customer_ids,negative_quantity,zero_price,negative_price
SourceSheet,,,,,
Year 2009-2010,525461,107927,12326,3687,3
Year 2010-2011,541910,135080,10624,2515,2


In [17]:
summary = {
    "Total rows": len(df),
    "Duplicate rows": df.duplicated().sum(),
    "Missing Customer IDs": df["Customer ID"].isna().sum(),
    "Missing Descriptions": df["Description"].isna().sum(),
    "Negative Quantity rows": (df["Quantity"] < 0).sum(),
    "Zero Price rows": (df["Price"] == 0).sum(),
    "Negative Price rows": (df["Price"] < 0).sum(),
    "Cancelled Invoice rows": df["Invoice"].astype(str).str.startswith("C", na=False).sum()
}

pd.Series(summary)

Total rows                1067371
Duplicate rows              12133
Missing Customer IDs       243007
Missing Descriptions         4382
Negative Quantity rows      22950
Zero Price rows              6202
Negative Price rows             5
Cancelled Invoice rows      19494
dtype: int64

In [19]:
cancelled_mask = df["Invoice"].astype(str).str.startswith("C", na=False)
negative_qty_mask = df["Quantity"] < 0

print("Cancelled + negative quantity:",
      (cancelled_mask & negative_qty_mask).sum())

print("Cancelled but quantity not negative:",
      (cancelled_mask & ~negative_qty_mask).sum())

print("Negative quantity but invoice not cancelled:",
      (~cancelled_mask & negative_qty_mask).sum())

Cancelled + negative quantity: 19493
Cancelled but quantity not negative: 1
Negative quantity but invoice not cancelled: 3457


In [18]:
df[df["Price"] < 0][
    ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate",
     "Price", "Customer ID", "Country", "SourceSheet"]
]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom,Year 2009-2010
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom,Year 2009-2010
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom,Year 2009-2010
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,Year 2010-2011
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,Year 2010-2011


## Cleaning Plan

After profiling the raw dataset, I decided on the following cleaning approach:

- Keep the original Excel workbook unchanged.
- Combine both yearly sheets into one dataset while keeping `SourceSheet` for traceability.
- Remove exact duplicate rows.
- Treat invoices beginning with `C` as cancelled/returned transactions and keep them separate from normal sales analysis.
- Investigate negative Quantity values that are not linked to cancelled invoices before deciding whether to remove or classify them as returns.
- Remove or isolate invalid negative Price records after checking them individually.
- Investigate zero-price transactions before deciding whether they are genuine free items, adjustments, or invalid records.
- Keep transactions with missing Customer ID for overall sales analysis where possible, but exclude them from customer-level analysis.
- Remove rows with missing Description when they cannot support product-level analysis.
- Preserve a cleaned transaction dataset and, where useful, separate returns/cancellations for later analysis.